# Notebook 5: 2D 전술 시각화 (v0.5)

시뮬레이션 결과를 2D 전투공간 위에 시각화합니다.
- 에이전트 배치 및 위협 이동 궤적
- 센서 탐지 범위, 사수 교전 범위
- 킬체인 이벤트 타임라인
- Linear C2 vs Kill Web 아키텍처 비교 스냅샷
- 전체 시뮬레이션 리플레이 애니메이션

In [ ]:
import matplotlib
matplotlib.use('Agg')  # 노트북 호환
%matplotlib inline

import matplotlib.pyplot as plt
from modules.model import AirDefenseModel
from modules.viz import TacticalVisualizer

print('모듈 로드 완료')

## 1. 시뮬레이션 실행 (스냅샷 기록 모드)

In [ ]:
# 시나리오 1, 양쪽 아키텍처 실행 (record_snapshots=True)
results = {}
for arch in ['linear', 'killweb']:
    m = AirDefenseModel(
        architecture=arch,
        scenario='scenario_1_saturation',
        seed=42,
        record_snapshots=True
    )
    results[arch] = m.run_full()
    metrics = results[arch]['metrics']
    print(f"{arch}: 누출률={metrics['leaker_rate']:.1f}%, "
          f"S2S={metrics['sensor_to_shooter_time']['mean']:.1f}s, "
          f"성공률={metrics['engagement_success_rate']:.1f}%, "
          f"스냅샷={len(results[arch]['snapshots'])}프레임")

## 2. 정적 스냅샷 — 주요 시점 비교

In [ ]:
# Kill Web 주요 시점 4패널
viz_kw = TacticalVisualizer(results['killweb'])
n_frames = len(results['killweb']['snapshots'])
time_indices = [0, n_frames // 4, n_frames // 2, n_frames - 1]

fig, axes = plt.subplots(2, 2, figsize=(20, 16))
for ax, idx in zip(axes.flatten(), time_indices):
    viz_kw.render_frame(idx, ax=ax)

plt.suptitle('Kill Web — 시나리오 1 주요 시점', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. 아키텍처 비교 — 동일 시점 Linear vs Kill Web

In [ ]:
# 중간 시점에서 비교
mid_frame = min(len(results['linear']['snapshots']),
                len(results['killweb']['snapshots'])) // 2

viz = TacticalVisualizer(results['killweb'])
fig = viz.snapshot_comparison(
    mid_frame,
    [results['linear'], results['killweb']],
    labels=['Linear C2', 'Kill Web']
)
plt.suptitle(f'아키텍처 비교 (프레임 {mid_frame})', fontsize=14, y=1.02)
plt.show()

## 4. 이벤트 타임라인

In [ ]:
# 양쪽 아키텍처 이벤트 타임라인
for arch in ['linear', 'killweb']:
    viz = TacticalVisualizer(results[arch])
    fig = viz.event_timeline()
    plt.show()

## 5. 전체 애니메이션 (GIF 저장)

In [ ]:
# Kill Web 애니메이션 생성 및 GIF 저장
viz_kw = TacticalVisualizer(results['killweb'], figsize=(12, 8))
anim = viz_kw.animate(interval_ms=300, save_path='output/killweb_s1_replay.gif')
print('애니메이션 저장 완료: output/killweb_s1_replay.gif')

## 6. COP 품질 비교 (v0.5)

Kill Web은 센서 융합으로 위치 오차가 감소하고,
아군 상태 공유로 사수 재할당이 최적화됩니다.

In [ ]:
# COP 품질 분석: Kill Web의 센서 융합 효과
from modules.config import ENGAGEMENT_POLICY
import math

base_error = ENGAGEMENT_POLICY['tracking_position_error_std']
print(f'기본 추적 오차: {base_error} km')
print(f'센서 1개: {base_error:.3f} km')
print(f'센서 2개: {base_error / math.sqrt(2):.3f} km')
print(f'센서 3개: {base_error / math.sqrt(3):.3f} km')
print(f'센서 4개: {base_error / math.sqrt(4):.3f} km')
print(f'\nLinear C2: 단일 센서 → 오차 {base_error} km 유지')
print(f'Kill Web: 다중 센서 융합 → 오차 감소 → Pk 보너스')

## 7. 전 시나리오 최종 상태 비교

In [ ]:
# 전 시나리오 실행 및 최종 스냅샷 비교 그리드
scenarios = [
    'scenario_1_saturation',
    'scenario_2_complex',
    'scenario_3_ew_light',
    'scenario_3_ew_heavy',
]

fig, axes = plt.subplots(len(scenarios), 2, figsize=(20, 8 * len(scenarios)))

for i, sc in enumerate(scenarios):
    for j, arch in enumerate(['linear', 'killweb']):
        m = AirDefenseModel(
            architecture=arch, scenario=sc, seed=42,
            record_snapshots=True
        )
        r = m.run_full()
        viz = TacticalVisualizer(r)
        # 최종 프레임
        last = len(r['snapshots']) - 1
        viz.render_frame(last, ax=axes[i][j])
        metrics = r['metrics']
        axes[i][j].set_title(
            f"{arch.upper()} — {sc}\n"
            f"누출={metrics['leaker_rate']:.1f}% "
            f"성공={metrics['engagement_success_rate']:.1f}%",
            fontsize=10
        )

plt.suptitle('전 시나리오 최종 상태 비교 (v0.5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()